In [60]:
import pandas as pd
import networkx as nx
from math import radians, sin, cos, sqrt, atan2, isnan
from shapely.geometry import Point, Polygon
from shapely.errors import GEOSException
import pyproj # Para la transformación de coordenadas
from math import isnan
from collections import defaultdict


## Importamos ficheros necesarios

In [61]:
# Cargar los archivos GTFS en DataFrames de pandas
try:
    routes_df = pd.read_csv(r'C:\Users\fbetancourt\Documents\GitHub\Tesis\Datasets\gtfs_amg_20240312\Datos\routes.csv')
    trips_df = pd.read_csv(r'C:\Users\fbetancourt\Documents\GitHub\Tesis\Datasets\gtfs_amg_20240312\Datos\trips.csv')
    stop_times_df = pd.read_csv(r'C:\Users\fbetancourt\Documents\GitHub\Tesis\Datasets\gtfs_amg_20240312\Datos\stop_times.csv')
    stops_df = pd.read_csv(r'C:\Users\fbetancourt\Documents\GitHub\Tesis\Datasets\gtfs_amg_20240312\Datos\stops.csv')
except FileNotFoundError as e:
    print(f"Error: Asegúrate de que los archivos CSV ({e.filename}) estén en el mismo directorio que el notebook.")
    # Detener la ejecución si los archivos no se encuentran
    raise

In [62]:
# Imprimir la cabecera de cada DataFrame para verificar la carga
print("routes_df head:")
print(routes_df.head())
print("\ntrips_df head:")
print(trips_df.head())
print("\nstop_times_df head:")
print(stop_times_df.head())
print("\nstops_df head:")
print(stops_df.head())

routes_df head:
  route_id agency_id route_short_name                        route_long_name  \
0      C01       BUS              C01           Troncal C01 - 39A "Poniente"   
1      C02       BUS              C02                      Troncal C02 - 45A   
2      C03       BUS              C03  Troncal C03 - 61 "Mercado de Abastos"   
3      C04       BUS              C04                      Troncal C04 - 110   
4      C05       BUS              C05                      Troncal C05 - 136   

   route_type route_url route_color route_text_color  
0           3       NaN      4C8D2B           FFFFFF  
1           3       NaN      4C8D2B           FFFFFF  
2           3       NaN      4C8D2B           FFFFFF  
3           3       NaN      4C8D2B           FFFFFF  
4           3       NaN      4C8D2B           FFFFFF  

trips_df head:
                     trip_id route_id service_id         trip_headsign  \
0  C01_route_1_0000001_16500      C01          D  Centro Metropolitano   
1  C01_ro

In [63]:
# Fusionar los DataFrames
# a. Fusionar stop_times_df con trips_df en 'trip_id'
merged_df = pd.merge(stop_times_df, trips_df, on='trip_id', how='left')
# b. Fusionar el resultado con routes_df en 'route_id'
merged_df = pd.merge(merged_df, routes_df, on='route_id', how='left')
# c. Fusionar el resultado con stops_df en 'stop_id'
merged_df = pd.merge(merged_df, stops_df, on='stop_id', how='left')

print("\nmerged_df head after all merges:")
print(merged_df.head())
print("\nColumnas en merged_df:")
print(merged_df.columns)



merged_df head after all merges:
                     trip_id           stop_id  stop_sequence stop_headsign  \
0  C01_route_1_0000001_16500  mxc_C01_P_STP_01              0      Libertad   
1  C01_route_1_0000001_16500  mxc_C01_P_STP_02              1  Calle Ocampo   
2  C01_route_1_0000001_16500  mxc_C01_P_STP_03              2  Calle Parral   
3  C01_route_1_0000001_16500  mxc_C01_P_STP_04              3  Calle Parral   
4  C01_route_1_0000001_16500  mxc_C01_P_STP_05              4  Calle Parral   

  arrival_time departure_time  timepoint route_id service_id  \
0     04:35:00       04:35:00          0      C01          D   
1     04:36:32       04:36:32          0      C01          D   
2     04:37:18       04:37:18          0      C01          D   
3     04:38:33       04:38:33          0      C01          D   
4     04:39:38       04:39:38          0      C01          D   

          trip_headsign  ...  agency_id route_short_name  \
0  Centro Metropolitano  ...        BUS       

## Creamos un grafo direccionado

In [64]:
# Crear un grafo dirigido vacío
G = nx.DiGraph()

# Agrupar merged_df por 'trip_id'
# Asegurarse de que las columnas necesarias existen antes de agrupar
required_columns = ['trip_id', 'stop_id', 'stop_sequence', 'route_id', 'stop_name', 'stop_lat', 'stop_lon']
for col in required_columns:
    if col not in merged_df.columns:
        print(f"Error: La columna '{col}' no se encuentra en merged_df después de las fusiones.")
        # Detener la ejecución si falta una columna crítica
        raise KeyError(f"Columna '{col}' faltante.")

grouped_trips = merged_df.groupby('trip_id')

In [65]:
# Para cada viaje (trip):
for trip_id, trip_data in grouped_trips:
    # a. Ordenar las paradas por 'stop_sequence'
    trip_data_sorted = trip_data.sort_values(by='stop_sequence')

    # b. Iterar a través de las paradas ordenadas y añadir aristas dirigidas
    #    desde la parada actual a la siguiente parada en la secuencia.
    stops_in_trip = trip_data_sorted['stop_id'].tolist()
    stop_details = trip_data_sorted.set_index('stop_id')

    for i in range(len(stops_in_trip) - 1):
        from_stop_id = stops_in_trip[i]
        to_stop_id = stops_in_trip[i+1]

        # c. Usar 'stop_id' para los nodos del grafo.
        # d. Añadir 'route_id', 'trip_id', 'stop_name', 'stop_lat', 'stop_lon'
        #    como atributos a los nodos si es la primera vez que se añade el nodo.

        # Añadir nodo de origen si no existe y sus atributos
        if not G.has_node(from_stop_id):
            details = stop_details.loc[from_stop_id]
            G.add_node(
                from_stop_id,
                route_id=details['route_id'],
                # trip_id=trip_id, # El trip_id puede variar si la parada es parte de múltiples viajes
                stop_name=details['stop_name'],
                stop_lat=details['stop_lat'],
                stop_lon=details['stop_lon']
            )

        # Añadir nodo de destino si no existe y sus atributos
        if not G.has_node(to_stop_id):
            details = stop_details.loc[to_stop_id]
            G.add_node(
                to_stop_id,
                route_id=details['route_id'],
                # trip_id=trip_id,
                stop_name=details['stop_name'],
                stop_lat=details['stop_lat'],
                stop_lon=details['stop_lon']
            )

        # Añadir la arista dirigida
        G.add_edge(from_stop_id, to_stop_id, trip_id=trip_id, route_id=stop_details.loc[from_stop_id, 'route_id'])

In [66]:
# Imprimir el número de nodos y aristas en el grafo G
print(f"\nGrafo construido.")
print(f"Número de nodos (paradas): {G.number_of_nodes()}")
print(f"Número de aristas (segmentos de viaje): {G.number_of_edges()}")

# Ejemplo de cómo acceder a los datos de un nodo
if G.number_of_nodes() > 0:
    sample_node = list(G.nodes())[0]
    print(f"\nDatos del nodo de ejemplo '{sample_node}': {G.nodes[sample_node]}")

# Ejemplo de cómo acceder a los datos de una arista
if G.number_of_edges() > 0:
    sample_edge = list(G.edges(data=True))[0]
    print(f"\nDatos de la arista de ejemplo: De '{sample_edge[0]}' a '{sample_edge[1]}' -> {sample_edge[2]}")#%%


Grafo construido.
Número de nodos (paradas): 10650
Número de aristas (segmentos de viaje): 12760

Datos del nodo de ejemplo 'MM_A03_1': {'route_id': 'MC-A03', 'stop_name': 'Terminal de la Ruta 110', 'stop_lat': 20.71663, 'stop_lon': -103.31998}

Datos de la arista de ejemplo: De 'MM_A03_1' a 'MM_A03_2' -> {'trip_id': 'A03_1', 'route_id': 'MC-A03'}


In [67]:
# Nombre del archivo de salida
output_filename = r"C:\Users\fbetancourt\Documents\GitHub\Tesis\QGIS\transporte_publico_grafo.gexf"

# Exportar el grafo a formato GEXF, que Gephi puede importar.
try:
    # Limpiar atributos si es necesario (ejemplo: convertir tipos no estándar a string)
    for node, data in G.nodes(data=True):
        for key, value in data.items():
            if not isinstance(value, (str, int, float, bool)):
                G.nodes[node][key] = str(value)

    for u, v, data in G.edges(data=True):
        for key, value in data.items():
            if not isinstance(value, (str, int, float, bool)):
                G.edges[u, v][key] = str(value)

    nx.write_gexf(G, output_filename)
    print(f"\nGrafo exportado exitosamente como '{output_filename}'.")
    print("Puedes importar este archivo directamente en Gephi.")
except NameError:
    print("Error: El grafo 'G' no fue encontrado. Asegúrate de ejecutar la celda anterior primero.")
except Exception as e:
    print(f"Ocurrió un error al exportar el grafo: {e}")


Grafo exportado exitosamente como 'C:\Users\fbetancourt\Documents\GitHub\Tesis\QGIS\transporte_publico_grafo.gexf'.
Puedes importar este archivo directamente en Gephi.


## Reducimos este grafo para que cada nodo sea un conjunto de paradas cercanas

In [68]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Calcula la distancia entre dos puntos en la Tierra (especificados en grados decimales)
    usando la fórmula de Haversine. Devuelve la distancia en metros.
    """
    if any(map(lambda x: x is None or isnan(x), [lat1, lon1, lat2, lon2])):
        return float('inf') # No se puede calcular la distancia si faltan coordenadas

    R = 6371000  # Radio de la Tierra en metros

    phi1 = radians(lat1)
    phi2 = radians(lat2)
    delta_phi = radians(lat2 - lat1)
    delta_lambda = radians(lon2 - lon1)

    a = sin(delta_phi / 2)**2 + cos(phi1) * cos(phi2) * sin(delta_lambda / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [69]:
# --- Inicio del proceso de unificación ---
nodes_with_coords = []
for node_id, data in G.nodes(data=True):
    if 'stop_lat' in data and 'stop_lon' in data and isinstance(data['stop_lat'], (int, float)) and isinstance(data['stop_lon'], (int, float)):
        nodes_with_coords.append({
            'id': node_id,
            'lat': data['stop_lat'],
            'lon': data['stop_lon'],
            'name': data.get('stop_name', 'Unknown')
        })
    else:
        print(f"Advertencia: Nodo {node_id} no tiene atributos 'stop_lat' o 'stop_lon'. Será ignorado en la unificación.")

Usando umbral de distancia: 100 metros.
Advertencia: Nodo MM_A16_70 no tiene atributos 'stop_lat' o 'stop_lon'. Será ignorado en la unificación.
Advertencia: Nodo mxc_C32V1_STP_53 no tiene atributos 'stop_lat' o 'stop_lon'. Será ignorado en la unificación.
Advertencia: Nodo mxc_C61_STP_135 no tiene atributos 'stop_lat' o 'stop_lon'. Será ignorado en la unificación.


In [70]:
# Algoritmo de clustering (componentes conectados basados en distancia)
visited_nodes = set()
clusters = [] # Lista de sets, donde cada set es un cluster de IDs de nodos originales

for i in range(len(nodes_with_coords)):
    node_i_id = nodes_with_coords[i]['id']
    if node_i_id in visited_nodes:
        continue

    current_cluster = {node_i_id}
    queue = [nodes_with_coords[i]] # Cola de diccionarios de nodos
    visited_nodes.add(node_i_id)

    head = 0
    while head < len(queue):
        current_node_in_q_dict = queue[head]
        head += 1

        for j in range(len(nodes_with_coords)): # Comparar con todos los demás nodos
            node_j_dict = nodes_with_coords[j]
            node_j_id = node_j_dict['id']

            if node_j_id not in visited_nodes:
                dist = haversine(current_node_in_q_dict['lat'], current_node_in_q_dict['lon'],
                                 node_j_dict['lat'], node_j_dict['lon'])

                if dist <= distance_threshold_meters:
                    visited_nodes.add(node_j_id)
                    current_cluster.add(node_j_id)
                    queue.append(node_j_dict)
    clusters.append(current_cluster)

In [71]:
# Crear mapeo de nodos antiguos a nuevos y construir el grafo unificado
node_mapping = {} # old_node_id -> new_node_id
G_unified = nx.DiGraph()
unified_node_counter = 0

print(f"\nSe encontraron {len(clusters)} clústeres/nodos individuales.")

for cluster in clusters:
    original_ids_in_cluster = list(cluster)

    if len(cluster) == 1:
        old_node_id = original_ids_in_cluster[0]
        node_mapping[old_node_id] = old_node_id
        if G.has_node(old_node_id): # Asegurarse que el nodo existe en G
            G_unified.add_node(old_node_id, **G.nodes[old_node_id])
    else:
        new_node_id = f"unified_cluster_{unified_node_counter}"
        unified_node_counter += 1

        sum_lat, sum_lon = 0, 0
        num_valid_coords = 0
        original_names = []

        for old_id in original_ids_in_cluster:
            node_data = G.nodes[old_id]
            # Usar las coordenadas almacenadas en G, que deberían ser las correctas
            lat = node_data.get('stop_lat')
            lon = node_data.get('stop_lon')
            if lat is not None and lon is not None and not (isnan(lat) or isnan(lon)):
                sum_lat += lat
                sum_lon += lon
                num_valid_coords +=1
            original_names.append(node_data.get('stop_name', old_id))
            node_mapping[old_id] = new_node_id

        avg_lat = sum_lat / num_valid_coords if num_valid_coords > 0 else None
        avg_lon = sum_lon / num_valid_coords if num_valid_coords > 0 else None

        # Atributos del nodo unificado
        unified_node_attrs = {
            'stop_lat': avg_lat,
            'stop_lon': avg_lon,
            'original_stop_ids': original_ids_in_cluster,
            'num_original_stops': len(original_ids_in_cluster),
            'stop_name': f"Cluster ({len(original_ids_in_cluster)} paradas): {', '.join(original_names[:2])}{'...' if len(original_names) > 2 else ''}",
            'type': 'unified_cluster'
        }
        G_unified.add_node(new_node_id, **unified_node_attrs)


Se encontraron 4203 clústeres/nodos individuales.


In [72]:
# Reconstruir aristas en el grafo unificado, agregando atributos
num_original_edges = G.number_of_edges()
num_unified_edges_created = 0

for u, v, edge_attrs in G.edges(data=True):
    new_u = node_mapping.get(u)
    new_v = node_mapping.get(v)

    if new_u is None or new_v is None:
        # Esto no debería suceder si todos los nodos de G se procesaron
        print(f"Advertencia: No se encontró mapeo para la arista ({u}, {v}). Se omite.")
        continue

    if new_u != new_v: # Evitar auto-bucles si un clúster se conecta a sí mismo (raro aquí)
        if G_unified.has_edge(new_u, new_v):
            # La arista ya existe, agregar/actualizar atributos
            current_edge_data = G_unified.edges[new_u, new_v]

            # Incrementar el contador de aristas originales colapsadas
            current_edge_data['collapsed_edge_count'] = current_edge_data.get('collapsed_edge_count', 0) + 1

            for key, value in edge_attrs.items():
                if key not in current_edge_data:
                    current_edge_data[key] = [] # Inicializa como lista si no está

                # Asegurarse de que es una lista
                if not isinstance(current_edge_data[key], list):
                    current_edge_data[key] = [current_edge_data[key]]

                # Agregar valor si no es duplicado
                if value not in current_edge_data[key]:
                    current_edge_data[key].append(value)
        else:
            # Nueva arista, inicializa atributos como listas
            attrs_to_add = {key: [value] for key, value in edge_attrs.items()}

            # Inicializar el contador de aristas originales colapsadas
            attrs_to_add['collapsed_edge_count'] = 1

            G_unified.add_edge(new_u, new_v, **attrs_to_add)
            num_unified_edges_created +=1

Advertencia: No se encontró mapeo para la arista (MM_A16_59, MM_A16_70). Se omite.
Advertencia: No se encontró mapeo para la arista (MM_A16_70, MM_A16_61). Se omite.
Advertencia: No se encontró mapeo para la arista (MM_A16_70, MM_A16_72). Se omite.
Advertencia: No se encontró mapeo para la arista (MM_A16_69, MM_A16_70). Se omite.
Advertencia: No se encontró mapeo para la arista (mxc_C06_STP_116, mxc_C61_STP_135). Se omite.
Advertencia: No se encontró mapeo para la arista (mxc_C28_STP_57, mxc_C32V1_STP_53). Se omite.
Advertencia: No se encontró mapeo para la arista (mxc_C32V1_STP_50, mxc_C32V1_STP_53). Se omite.
Advertencia: No se encontró mapeo para la arista (mxc_C32V1_STP_53, mxc_C36_STP_62). Se omite.
Advertencia: No se encontró mapeo para la arista (mxc_C32V1_STP_53, mxc_C32V1_STP_55). Se omite.
Advertencia: No se encontró mapeo para la arista (mxc_C32V1_STP_53, mxc_C37V1_STP_70). Se omite.
Advertencia: No se encontró mapeo para la arista (mxc_C36_STP_62, mxc_C32V1_STP_53). Se omit

In [73]:
# Imprimir información sobre el nuevo grafo
print(f"\nGrafo original: {G.number_of_nodes()} nodos, {num_original_edges} aristas.")
print(f"Grafo unificado: {G_unified.number_of_nodes()} nodos, {G_unified.number_of_edges()} aristas.")


unified_output_filename = r"C:\Users\fbetancourt\Documents\GitHub\Tesis\QGIS\transporte_publico_grafo_unificado.gexf"
try:
    # Limpiar atributos para serialización GEXF
    for node_id, data_node in G_unified.nodes(data=True):
        for key, value in data_node.items():
            if isinstance(value, list): # Convertir listas a string para GEXF
                G_unified.nodes[node_id][key] = '; '.join(map(str, value))
            elif not isinstance(value, (str, int, float, bool)) and value is not None:
                G_unified.nodes[node_id][key] = str(value)

    for u_edge, v_edge, data_edge in G_unified.edges(data=True):
        for key, value in data_edge.items():
            if isinstance(value, list): # Convertir listas a string para GEXF
                G_unified.edges[u_edge, v_edge][key] = '; '.join(map(str, value))
            elif not isinstance(value, (str, int, float, bool)) and value is not None:
                G_unified.edges[u_edge, v_edge][key] = str(value)

    nx.write_gexf(G_unified, unified_output_filename)
    print(f"Grafo unificado exportado como '{unified_output_filename}'.")
except Exception as e:
    print(f"Ocurrió un error al exportar el grafo unificado: {e}")
    print("Puede que necesites ajustar manualmente los tipos de datos de los atributos si la exportación falla.")


Grafo original: 10650 nodos, 12760 aristas.
Grafo unificado: 4203 nodos, 8583 aristas.
Grafo unificado exportado como 'C:\Users\fbetancourt\Documents\GitHub\Tesis\QGIS\transporte_publico_grafo_unificado.gexf'.


## Etiquetado de nodos por centralidad

In [75]:
POLYGON_CRS_EPSG = 'EPSG:32613'
NODE_CRS_EPSG = 'EPSG:4326'

# Nombres de los archivos CSV
coords_filename = r'C:\Users\fbetancourt\Documents\GitHub\Tesis\Datasets\shapes_imeplan\coordenadas_con_indice.csv'
attributes_filename = r'C:\Users\fbetancourt\Documents\GitHub\Tesis\Datasets\shapes_imeplan\atributos_centralidad.csv'

target_graph_name = ''
if 'G_unified' in globals() and G_unified is not None:
    target_graph = G_unified
    target_graph_name = 'G_unified'
    print("Usando el grafo 'G_unified' para etiquetar nodos.")
elif 'G' in globals() and G is not None:
    target_graph = G
    target_graph_name = 'G'
    print("Usando el grafo 'G' para etiquetar nodos.")
else:
    target_graph = None
    print("Error: No se encontró ningún grafo ('G' o 'G_unified') en memoria. Ejecuta las celdas anteriores.")

Usando el grafo 'G_unified' para etiquetar nodos.


In [76]:
# --- Funciones Auxiliares ---
def load_centrality_attributes(filename):
    """Carga los atributos de centralidad y los devuelve como un diccionario."""
    try:
        attr_df = pd.read_csv(filename)
        # Asegurarse de que indice_centralidad sea del tipo correcto para el merge/map
        attr_df['indice_centralidad'] = attr_df['indice_centralidad'].astype(int)
        return attr_df.set_index('indice_centralidad').to_dict('index')
    except FileNotFoundError:
        print(f"Error: Archivo de atributos '{filename}' no encontrado.")
        return None
    except Exception as e:
        print(f"Error al cargar atributos de centralidad: {e}")
        return None

def load_and_build_polygons(filename, attributes_dict):
    """Carga coordenadas, construye polígonos y asocia atributos."""
    polygons_data = {}
    try:
        coords_df = pd.read_csv(filename)
        # Asegurarse de que indice_centralidad sea del tipo correcto
        coords_df['indice_centralidad'] = coords_df['indice_centralidad'].astype(int)

        for idx_centralidad, group in coords_df.groupby('indice_centralidad'):
            # Las columnas se llaman 'longitud' y 'latitud' en el CSV,
            # pero son X e Y del sistema proyectado.
            points = group.sort_values(by='orden_punto')[['longitud', 'latitud']].values.tolist()

            if len(points) < 3:
                # print(f"Advertencia: Insuficientes puntos ({len(points)}) para formar polígono para indice_centralidad {idx_centralidad}. Se omite.")
                continue

            try:
                poly = Polygon(points)
                if not poly.is_valid:
                    # print(f"Advertencia: Geometría inválida para polígono {idx_centralidad}. Intentando buffer(0).")
                    poly = poly.buffer(0) # Intento de corrección común
                    if not poly.is_valid or poly.is_empty:
                        # print(f"Error: Polígono {idx_centralidad} sigue inválido o vacío después de buffer(0). Se omite.")
                        continue

                attrs = attributes_dict.get(idx_centralidad, {})
                polygons_data[idx_centralidad] = {
                    'polygon': poly,
                    'nombre_centralidad': attrs.get('nombre', f'Desconocido_{idx_centralidad}'),
                    'tipo_centralidad': attrs.get('tipo', 'Desconocido')
                }
            except GEOSException as e:
                print(f"Error de GEOS al crear polígono para {idx_centralidad}: {e}. Se omite.")
            except Exception as e_inner:
                print(f"Error inesperado al crear polígono para {idx_centralidad}: {e_inner}. Se omite.")

        return polygons_data
    except FileNotFoundError:
        print(f"Error: Archivo de coordenadas '{filename}' no encontrado.")
        return None
    except Exception as e:
        print(f"Error al cargar o construir polígonos: {e}")
        return None

In [77]:
# --- Lógica Principal ---
print(f"Iniciando proceso de etiquetado para el grafo '{target_graph_name}'.")
print(f"Coordenadas de nodos del grafo (EPSG:{NODE_CRS_EPSG}) se transformarán a ({POLYGON_CRS_EPSG}).")

centrality_attributes = load_centrality_attributes(attributes_filename)
polygons_with_attributes = None
if centrality_attributes:
    polygons_with_attributes = load_and_build_polygons(coords_filename, centrality_attributes)

if centrality_attributes and polygons_with_attributes:
    # Preparar transformador de coordenadas
    try:
        transformer = pyproj.Transformer.from_crs(NODE_CRS_EPSG, POLYGON_CRS_EPSG, always_xy=True)
    except Exception as e:
        print(f"Error al crear el transformador de coordenadas: {e}")
        print("Asegúrate de que los códigos EPSG son válidos y pyproj está instalado correctamente.")
        transformer = None

    if transformer:
        nodes_tagged_count = 0
        nodes_without_coords_count = 0
        nodes_outside_any_polygon_count = 0

        for node_id, data in target_graph.nodes(data=True):
            # Inicializar atributos en caso de que no se encuentre polígono
            data['indice_centralidad_poligono'] = None
            data['nombre_centralidad'] = None
            data['tipo_centralidad'] = None

            node_lat = data.get('stop_lat')
            node_lon = data.get('stop_lon')

            if node_lat is None or node_lon is None or isnan(node_lat) or isnan(node_lon):
                # print(f"Advertencia: Nodo {node_id} no tiene coordenadas válidas. Se omite la asignación de centralidad.")
                nodes_without_coords_count +=1
                continue

            try:
                # Transformar coordenadas del nodo al sistema de los polígonos
                # pyproj espera (x, y) o (lon, lat)
                node_x_proj, node_y_proj = transformer.transform(node_lon, node_lat)
                node_point_proj = Point(node_x_proj, node_y_proj)

                found_polygon = False
                for idx_centralidad, poly_data in polygons_with_attributes.items():
                    if poly_data['polygon'].contains(node_point_proj):
                        data['indice_centralidad_poligono'] = idx_centralidad
                        data['nombre_centralidad'] = poly_data['nombre_centralidad']
                        data['tipo_centralidad'] = poly_data['tipo_centralidad']
                        nodes_tagged_count += 1
                        found_polygon = True
                        break # Asumimos que un nodo pertenece a un solo polígono

                if not found_polygon:
                    nodes_outside_any_polygon_count += 1

            except Exception as e_transform:
                # El nodo mantendrá los atributos de centralidad como None
                nodes_without_coords_count +=1 # Contar como no procesado correctamente

        print(f"\nProceso de etiquetado completado.")
        print(f"Nodos etiquetados con información de centralidad: {nodes_tagged_count}")
        print(f"Nodos sin coordenadas válidas o con error de transformación: {nodes_without_coords_count}")
        print(f"Nodos que no cayeron dentro de ningún polígono de centralidad: {nodes_outside_any_polygon_count}")

        # Ejemplo de cómo ver los datos de un nodo etiquetado (si alguno fue etiquetado)
        if nodes_tagged_count > 0:
            sample_tagged_node_id = None
            for node_id, data_node in target_graph.nodes(data=True):
                if data_node.get('indice_centralidad_poligono') is not None:
                    sample_tagged_node_id = node_id
                    break
            if sample_tagged_node_id:
                print(f"\nEjemplo de datos para nodo etiquetado '{sample_tagged_node_id}':")
                print(f"  - Lat: {target_graph.nodes[sample_tagged_node_id].get('stop_lat')}")
                print(f"  - Lon: {target_graph.nodes[sample_tagged_node_id].get('stop_lon')}")
                print(f"  - Indice Centralidad: {target_graph.nodes[sample_tagged_node_id].get('indice_centralidad_poligono')}")
                print(f"  - Nombre Centralidad: {target_graph.nodes[sample_tagged_node_id].get('nombre_centralidad')}")
                print(f"  - Tipo Centralidad: {target_graph.nodes[sample_tagged_node_id].get('tipo_centralidad')}")
                if target_graph.nodes[sample_tagged_node_id].get('original_stop_ids'):
                    print(f"  - Original Stop IDs (si es unificado): {target_graph.nodes[sample_tagged_node_id].get('original_stop_ids')}")
    else:
        print("No se pudo crear el transformador de coordenadas. El etiquetado no se realizará.")

elif not centrality_attributes:
    print("No se cargaron atributos de centralidad. No se puede continuar.")
elif not polygons_with_attributes:
    print("No se construyeron polígonos. No se puede continuar.")

Iniciando proceso de etiquetado para el grafo 'G_unified'.
Coordenadas de nodos del grafo (EPSG:EPSG:4326) se transformarán a (EPSG:32613).

Proceso de etiquetado completado.
Nodos etiquetados con información de centralidad: 4122
Nodos sin coordenadas válidas o con error de transformación: 0
Nodos que no cayeron dentro de ningún polígono de centralidad: 81

Ejemplo de datos para nodo etiquetado 'unified_cluster_0':
  - Lat: 20.716172
  - Lon: -103.32007349999999
  - Indice Centralidad: 5
  - Nombre Centralidad: Huentitán
  - Tipo Centralidad: Centralidad Periférica
  - Original Stop IDs (si es unificado): MM_023; mxc_C11_STP_54; mxc_C16_STP_41; mxc_C138_STP_01; mxc_C138_STP_57; MM_A03_1; mxc_C11_STP_55; MM_032; mxc_C11_STP_08; mxc_C16_STP_92


In [78]:
active_graph = None
active_graph_name = ""

output_gexf_filename = output_gexf_filename = r"C:\Users\fbetancourt\Documents\GitHub\Tesis\QGIS\transporte_publico_grafo_unificado_etiquetado.gexf"

active_graph = target_graph
active_graph_name = target_graph_name # 'target_graph_name' también debería venir de la celda anterior

if active_graph is not None:
    for node_id in active_graph.nodes():
        for attr_key, attr_value in list(active_graph.nodes[node_id].items()): # Usar list() para permitir la modificación
            if attr_value is None:
                active_graph.nodes[node_id][attr_key] = ""  # Reemplazar None con una cadena vacía

    # Opcional: Iterar sobre todas las aristas y sus atributos para reemplazar None
    for u, v in active_graph.edges():
        for attr_key, attr_value in list(active_graph.edges[u, v].items()): # Usar list() para permitir la modificación
            if attr_value is None:
                active_graph.edges[u, v][attr_key] = "" # Reemplazar None con una cadena vacía

    # Intentar escribir el archivo GEXF de nuevo con el grafo corregido
    try:
        # Crear una copia para no modificar el grafo en memoria si es necesario
        # G_export = active_graph.copy() # Opcional, si la limpieza es destructiva

        # Limpiar atributos para serialización GEXF (convertir listas a string)
        # Es importante si hay atributos de tipo lista, comunes en G_unified para aristas.
        for node_id, data_node in active_graph.nodes(data=True):
            for key, value in data_node.items():
                if isinstance(value, list):
                    active_graph.nodes[node_id][key] = '; '.join(map(str, value))
                elif not isinstance(value, (str, int, float, bool)) and value is not None:
                    active_graph.nodes[node_id][key] = str(value)

        for u_edge, v_edge, data_edge in active_graph.edges(data=True):
            for key, value in data_edge.items():
                if isinstance(value, list):
                    active_graph.edges[u_edge, v_edge][key] = '; '.join(map(str, value))
                elif not isinstance(value, (str, int, float, bool)) and value is not None:
                    active_graph.edges[u_edge, v_edge][key] = str(value)

        nx.write_gexf(active_graph, output_gexf_filename)
        print(f"Grafo '{active_graph_name}' exportado exitosamente como '{output_gexf_filename}'.")
        print("Puedes importar este archivo directamente en Gephi.")

    except Exception as e:
        print(f"Ocurrió un error al exportar el grafo '{active_graph_name}': {e}")
        print("Asegúrate de que el grafo exista y los atributos sean compatibles con GEXF.")
else:
    print("Error: No se pudo determinar el grafo activo ('G' o 'G_unified') para exportar.")
    print("Asegúrate de haber ejecutado las celdas anteriores correctamente.")

Grafo 'G_unified' exportado exitosamente como 'C:\Users\fbetancourt\Documents\GitHub\Tesis\QGIS\transporte_publico_grafo_unificado_etiquetado.gexf'.
Puedes importar este archivo directamente en Gephi.


## Colapsar grafo por Centralidad de Nodos

En esta sección, tomaremos el grafo `active_graph` (que se asume ya tiene calculado un atributo de centralidad para sus nodos) y crearemos un nuevo grafo. En este nuevo grafo:
- Los nodos originales que comparten el mismo valor de centralidad se agruparán en un único "supernodo".
- Los nodos que no tengan un valor de centralidad definido (o sea `None`) serán descartados.
- Las aristas se remapearán entre los nuevos supernodos.
- El grafo resultante se guardará en formato GEXF, compatible con Gephi.


In [79]:
print(f"Creando grafo colapsado por centralidad a partir de '{active_graph_name}'...")

G_centrality_collapsed = nx.DiGraph() # O nx.Graph() si no es dirigido

# Agrupar nodos del active_graph por su 'indice_centralidad_poligono'
# Usaremos 'indice_centralidad_poligono' como ID para los nodos colapsados.
# Los nodos sin centralidad (None) se agruparán bajo una clave especial.
NO_CENTRALITY_KEY = "SIN_CENTRALIDAD_ASIGNADA"

node_to_centrality_map = {} # Mapea old_node_id -> centrality_key
centrality_nodes_data = defaultdict(lambda: {"original_node_ids": [], "summed_attrs": defaultdict(float), "attrs_to_set": {}, "sum_lat": 0.0, "sum_lon": 0.0, "coord_count": 0})

for node_id, data in active_graph.nodes(data=True):
    centrality_idx = data.get('indice_centralidad_poligono')
    centrality_key = centrality_idx if pd.notna(centrality_idx) else NO_CENTRALITY_KEY # pd.notna maneja None, NaN
    if isinstance(centrality_key, float) and pd.isna(centrality_key): # Doble chequeo por si acaso
        centrality_key = NO_CENTRALITY_KEY

    node_to_centrality_map[node_id] = centrality_key
    centrality_nodes_data[centrality_key]['original_node_ids'].append(node_id)

    group_data = centrality_nodes_data[centrality_key] # Referencia al diccionario del grupo
    group_data['original_node_ids'].append(node_id)

    # Sumar coordenadas si son válidas
    node_lat = data.get('stop_lat')
    node_lon = data.get('stop_lon')
    if node_lat is not None and node_lon is not None and not (isnan(node_lat) or isnan(node_lon)):
        group_data['sum_lat'] += node_lat
        group_data['sum_lon'] += node_lon
        group_data['coord_count'] += 1

    # Sumar atributos numéricos y establecer otros
    for attr_key, attr_value in data.items():
        if isinstance(attr_value, (int, float)) and not isinstance(attr_value, bool): # Sumar números (no booleanos)
            # Lista de atributos que NO deben sumarse directamente (ej. coordenadas)
            if attr_key not in ['stop_lat', 'stop_lon', 'latitud', 'longitud', 'indice_centralidad_poligono', 'orden_punto']:
                group_data['summed_attrs'][attr_key] += attr_value
        elif attr_key in ['nombre_centralidad', 'tipo_centralidad']: # Tomar estos del primer nodo (deberían ser consistentes)
            if attr_key not in group_data['attrs_to_set']:
                group_data['attrs_to_set'][attr_key] = attr_value

Creando grafo colapsado por centralidad a partir de 'G_unified'...


In [80]:
# Crear nodos en el grafo colapsado
for centrality_key, data in centrality_nodes_data.items():
    node_attrs = {
        "label": data['attrs_to_set'].get('nombre_centralidad', str(centrality_key)), # Usar nombre_centralidad como label
        "original_node_count": len(data['original_node_ids']),
    }
    if 'tipo_centralidad' in data['attrs_to_set']:
        node_attrs['tipo_centralidad'] = data['attrs_to_set']['tipo_centralidad']

    # Añadir los atributos sumados
    for summed_attr_key, summed_attr_value in data['summed_attrs'].items():
        node_attrs[summed_attr_key] = summed_attr_value

    # Calcular y añadir coordenadas promedio
    if data['coord_count'] > 0:
        node_attrs['stop_lat'] = data['sum_lat'] / data['coord_count']
        node_attrs['stop_lon'] = data['sum_lon'] / data['coord_count']
    else: # Si no hay coordenadas válidas, dejar como None o no añadir
        node_attrs['stop_lat'] = None
        node_attrs['stop_lon'] = None

    G_centrality_collapsed.add_node(centrality_key, **node_attrs)

In [81]:
# Crear aristas en el grafo colapsado
edge_aggregation = defaultdict(lambda: defaultdict(float))

for u, v, edge_data in active_graph.edges(data=True):
    u_centrality_key = node_to_centrality_map.get(u)
    v_centrality_key = node_to_centrality_map.get(v)

    if u_centrality_key is None or v_centrality_key is None:
        # Esto no debería ocurrir si todos los nodos están en el mapa
        print(f"Advertencia: No se encontró mapeo de centralidad para arista ({u}, {v}). Se omite.")
        continue

    current_edge_key = (u_centrality_key, v_centrality_key)

    # Incrementar el contador de aristas del active_graph que se agrupan aquí
    # Usamos .get para inicializar a 0 si la clave no existe aún
    edge_aggregation[current_edge_key]['num_active_graph_edges'] = \
        edge_aggregation[current_edge_key].get('num_active_graph_edges', 0) + 1


    # Sumar atributos numéricos de las aristas
    current_edge_key = (u_centrality_key, v_centrality_key)
    for attr_key, attr_value in edge_data.items():
        if isinstance(attr_value, (int, float)) and not isinstance(attr_value, bool):
            if attr_key == 'collapsed_edge_count':
                # Sumar específicamente la propiedad 'collapsed_edge_count'
                edge_aggregation[current_edge_key]['sum_original_collapsed_edges'] = \
                    edge_aggregation[current_edge_key].get('sum_original_collapsed_edges', 0) + attr_value
            else:
                # Sumar otros atributos numéricos genéricamente
                edge_aggregation[current_edge_key][attr_key] += attr_value

for (u_cen, v_cen), aggregated_attrs in edge_aggregation.items():
    if not G_centrality_collapsed.has_node(u_cen) or not G_centrality_collapsed.has_node(v_cen):
        print(f"Advertencia: Nodos de centralidad {u_cen} o {v_cen} no encontrados al crear arista. Omitiendo.")
        continue

    # Si no hay atributos agregados (ej. ninguna arista tenía numéricos, y no se añadió un 'weight' por defecto)
    # podemos establecer un peso de 1 o manejarlo como sea necesario.
    if not aggregated_attrs:
        aggregated_attrs['weight'] = 1.0 # Default weight si no hay otros atributos numéricos

    G_centrality_collapsed.add_edge(u_cen, v_cen, **aggregated_attrs)

print(f"Grafo colapsado por centralidad creado: {G_centrality_collapsed.number_of_nodes()} nodos, {G_centrality_collapsed.number_of_edges()} aristas.")

Grafo colapsado por centralidad creado: 54 nodos, 296 aristas.


In [82]:
# Guardar el grafo colapsado en GEXF
collapsed_output_gexf_filename = r"C:\Users\fbetancourt\Documents\GitHub\Tesis\QGIS\transporte_publico_grafo_unificado_etiquetado_colapsado.gexf"
try:
    # Limpieza de atributos para GEXF (similar a celdas anteriores)
    # GEXF no maneja bien todos los tipos de datos directamente (ej. None, listas complejas si no se serializaron)
    for node_id, data_node in G_centrality_collapsed.nodes(data=True):
        for key, value in list(data_node.items()): # Usar list(items()) para poder borrar durante la iteración
            if value is None: # GEXF puede tener problemas con None, convertir a string o quitar
                data_node[key] = "None" # o del data_node[key]
            elif isinstance(value, list):
                data_node[key] = '; '.join(map(str, value))
            elif not isinstance(value, (str, int, float, bool)):
                data_node[key] = str(value)

    for u_edge, v_edge, data_edge in G_centrality_collapsed.edges(data=True):
        for key, value in list(data_edge.items()):
            if value is None:
                data_edge[key] = "None" # o del data_edge[key]
            elif isinstance(value, list):
                data_edge[key] = '; '.join(map(str, value))
            elif not isinstance(value, (str, int, float, bool)):
                data_edge[key] = str(value)

    nx.write_gexf(G_centrality_collapsed, collapsed_output_gexf_filename)
    print(f"Grafo colapsado exportado exitosamente como '{collapsed_output_gexf_filename}'.")
except Exception as e:
    print(f"Ocurrió un error al exportar el grafo colapsado: {e}")

Grafo colapsado exportado exitosamente como 'C:\Users\fbetancourt\Documents\GitHub\Tesis\QGIS\transporte_publico_grafo_unificado_etiquetado_colapsado.gexf'.


In [83]:
nx.nodes(G_centrality_collapsed)

NodeView((5, 6, 37, 34, 35, 32, 38, 33, 65, 69, 21, 70, 41, 8, 7, 18, 9, 40, 30, 22, 43, 57, 56, 39, 17, 11, 36, 24, 42, '', 61, 26, 64, 60, 29, 52, 27, 19, 46, 62, 68, 44, 20, 13, 25, 23, 12, 31, 16, 1, 48, 63, 2, 15))